# MIDAS on Kaggle — CHAI attack against InternVL2.5-8B

Reproduces the CHAI visual sign-injection attack and evaluates the
activation-steering defense, at the model scale CHAI themselves used.

**Before running, set the session up:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (needs phone verification on your Kaggle account) |
| Persistence | Variables and files |

Two T4s give 32 GB of VRAM combined. The 8B model needs ~16 GB in float16, so it
is sharded across both with `--device-map auto`.

**Expected cost:** roughly 2–3 GPU-hours for the full attack evaluation, against
a weekly quota of about 30.

## 1. Check the hardware

Run this first. If it reports anything other than two T4s, stop and fix the
accelerator setting — everything below assumes this layout.

In [ ]:
import os
# Reduces fragmentation, which the first OOM on this hardware recommended.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, torch, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  gpu {i}: {torch.cuda.get_device_name(i)}  "
          f"sm_{major}{minor}  {total:.1f} GB")

# Gate on compute capability, as midas.models.internvl.resolve_dtype does.
# torch.cuda.is_bf16_supported() reports True on a T4 because recent PyTorch
# counts emulated bf16, but Turing has no bf16 tensor cores and the emulated
# path is slower than float16.
cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
torch_claim = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
bf16 = cc >= 8
fa2 = bf16
print(f"torch bf16 claim   : {torch_claim}  (counts emulation -- not what we use)")
print(f"native bfloat16    : {bf16}   -> dtype will be {'bfloat16' if bf16 else 'float16'}")
print(f"FlashAttention-2 ok: {fa2}   -> attention will run {'flash' if fa2 else 'eager'}")
if not fa2:
    print("  (expected on T4: sm_75 is Turing, FA2 needs sm_80+. Eager attention")
    print("   uses much more memory, so keep --max-tiles modest at first.)")

## 2. Install dependencies

Kaggle preinstalls PyTorch built against its own CUDA, so **do not reinstall
torch** — that risks breaking GPU support and wastes several minutes.

Two pins are load-bearing and cost us hours to diagnose locally:

- **`sentencepiece==0.2.1`.** Version 0.2.2 rejects InternVL's `tokenizer.model`
  as malformed even though its checksum matches the hub; `AutoTokenizer` swallows
  the error and returns `False`, which surfaces much later as `'bool' object has
  no attribute 'convert_tokens_to_ids'`. 0.1.99 also works but has no Python 3.13
  wheel, and Kaggle now runs 3.13 — pinning it makes pip try a source build,
  which fails and aborts the whole install.
- **No `numpy<2` pin.** Kaggle's preinstalled stack calls
  `numpy.dtypes.StringDType`, which only exists in numpy 2.0+. Forcing numpy 1.x
  makes `transformers` fail to import its model code with an error that names
  llama rather than numpy.
- **`transformers<4.50`.** InternVL2.5's remote code does not inherit from
  `GenerationMixin`, which 4.50 made mandatory. Kaggle ships 5.x, so this is a
  real downgrade and takes a minute.

The install is checked below rather than assumed: a failed pip run leaves the
preinstalled versions in place, and the notebook would then fail much later for
reasons that look unrelated.

In [ ]:
%pip install -q "transformers>=4.49,<4.50" "sentencepiece==0.2.1" protobuf qwen-vl-utils \
    "accelerate>=0.26" timm einops opencv-python-headless scikit-learn pytest

import importlib, sentencepiece, transformers, accelerate
for mod in (transformers, sentencepiece, accelerate):
    importlib.reload(mod)

print("transformers :", transformers.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("accelerate   :", accelerate.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
assert (major, minor) < (4, 50), (
    f"transformers {transformers.__version__} is too new; InternVL2.5's remote "
    "code needs <4.50. The pip step above must have failed -- scroll up."
)
assert sentencepiece.__version__ == "0.2.1", (
    f"sentencepiece {sentencepiece.__version__} installed, need 0.2.1. "
    "0.2.2 cannot read InternVL's tokenizer and fails confusingly later."
)
print("version checks passed")

## 3. Load the code

The code and CHAI's released artefacts arrive as an attached Kaggle Dataset, not
a git clone, so this notebook needs no GitHub repository and no access token.
Upload it with `python scripts/kaggle_deploy.py upload`.

The dataset is read-only, so it is copied into the working directory before the
editable install.

In [ ]:
import os, shutil, subprocess, glob
from pathlib import Path

INPUT = Path("/kaggle/input")
print("contents of /kaggle/input:")
if INPUT.exists():
    for entry in sorted(INPUT.iterdir()):
        inner = sorted(p.name for p in entry.iterdir())[:12] if entry.is_dir() else []
        print(f"  {entry.name}/  -> {inner}")
else:
    print("  (missing entirely -- no dataset is attached to this kernel)")

# Search recursively: a dataset may mount its payload one level deeper than
# expected depending on how the archive was packed.
# Datasets mount under /kaggle/input/datasets/<user>/<slug>; kernel outputs
# mount under /kaggle/input/<kernel-slug>/. The sweep kernel attaches both, and
# the main run saved its own working copy of this source tree as output -- so an
# unqualified search finds that stale copy instead of the uploaded dataset.
candidates = sorted(
    glob.glob("/kaggle/input/**/pyproject.toml", recursive=True),
    key=lambda p: (0 if "/kaggle/input/datasets/" in p else 1, p),
)
assert candidates, (
    "MIDAS source not found under /kaggle/input. Attach the dataset via "
    "Add Data -> Your Datasets, or re-run scripts/kaggle_deploy.py upload. "
    "If the dataset was created moments before this kernel started, it may "
    "still have been processing -- wait for `kaggle datasets status` to read "
    "ready, then push again."
)
SOURCE = Path(candidates[0]).parent
# Not "/kaggle/working/midas": Kaggle puts /kaggle/working on sys.path, so a
# directory of that name shadows the installed package and `import midas`
# resolves to an empty namespace package with __file__ None.
WORK = Path("/kaggle/working/midas_src")
print("source dataset:", SOURCE)

# /kaggle/input is read-only and `pip install -e` needs to write egg-info.
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

chai = WORK / "vendor" / "chai"
assert (chai / "training.json").exists(), (
    f"CHAI artefacts missing from the dataset (looked in {chai}); "
    "re-run the upload step"
)
frames = len(glob.glob(str(chai / "DriveLM_dataset/**/*.jpg"), recursive=True))
print(f"CHAI frames present: {frames} (expect 184)")
assert frames == 184, "frame count wrong -- the upload was incomplete"

%pip install -q -e .

# The editable install registers the console scripts (midas-eval and friends),
# which run as separate processes. It does not refresh this already-running
# kernel's import path, so point sys.path at the sources directly.
import importlib, sys, shutil as _shutil

sys.path.insert(0, str(WORK / "src"))
importlib.invalidate_caches()

import midas
assert midas.__file__, (
    "`midas` imported as a namespace package, so something on sys.path shadows "
    "it -- check for a directory of that name under /kaggle/working"
)
print("midas imported from", midas.__file__)

missing = [c for c in ("midas-eval", "midas-activations", "midas-probes")
           if _shutil.which(c) is None]
assert not missing, f"console scripts not on PATH: {missing}"
print("console scripts available")

## 4. Verify the data before spending GPU time

These checks are CPU-only and take seconds. They catch a bad clone before a
multi-hour run does.

In [ ]:
!python -m pytest tests -q

from midas import data
for split in ("train", "test"):
    scenes = data.load_scenes(split)
    missing = data.missing_frames(scenes)
    print(f"{split}: {len(scenes)} scenes, {len(missing)} missing frames")

from midas.tasks import drivelm
diffs = drivelm.verify_matches_upstream()
print("prompt matches CHAI source:", "yes" if not diffs else diffs)

## 5. Download the model

`OpenGVLab/InternVL2_5-8B` is about 16 GB. It goes to the HuggingFace cache rather than
`/kaggle/working`, which only has 20 GB and must also hold results.

To skip this download on later sessions, upload the cache once as a Kaggle
Dataset and attach it.

In [ ]:
from huggingface_hub import snapshot_download
import time

# Skipped when no selected stage uses InternVL: the 8B is ~16 GB, and a
# Qwen-only run would spend minutes fetching weights it never loads.
if False:
    t0 = time.time()
    path = snapshot_download(
        "OpenGVLab/InternVL2_5-8B",
        allow_patterns=["*.json", "*.py", "*.safetensors", "*.txt", "*.model"],
    )
    print(f"downloaded in {(time.time() - t0) / 60:.1f} min -> {path}")
    !du -sh {path}
else:
    print("skipping the InternVL download: no selected stage uses it")

### Shared names

Every stage below is independent and can be skipped, so the constants and
imports they share are defined once, here, rather than inside whichever stage
happened to need them first.

In [ ]:
import collections, glob, json, subprocess
from pathlib import Path

# The scale study and the option-order control both sweep these.
SIZES = ["OpenGVLab/InternVL2_5-1B", "OpenGVLab/InternVL2_5-2B",
         "OpenGVLab/InternVL2_5-4B"]

# Injection styles run on the 2B: it had 0% unreadable answers, where on the 8B
# a quarter of replies name both actions and would bury the signal.
STYLE_MODEL = "OpenGVLab/InternVL2_5-2B"
STYLES = ["plain", "authority", "override", "urgency"]

RESULTS = Path("/kaggle/working/results")

# The DriveLM scene set and the second model family. Both the evaluation stage
# and the steering stage need these, and either can run without the other, so
# they are resolved once here. Missing scenes are not an error yet -- only the
# stages that use them care.
DRIVELM_MODEL = "Qwen/Qwen2-VL-7B-Instruct"
DRIVELM_SPLIT = "drivelm"
# The strongest wording on this model at n=100: +25.0 points overall,
# McNemar p=0.0005, and +58.6 in the direction where the attack works. Both the
# steering stage and the single-direction replication use it.
ARGUS_STYLE = "plain"
_found = glob.glob("/kaggle/input/**/drivelm/scenes_clean200.json", recursive=True)
DRIVELM_SCENES = _found[0] if _found else None
print("DriveLM model :", DRIVELM_MODEL)

print("SIZES:", [s.split("-")[-1] for s in SIZES], " STYLES:", STYLES)
print("DriveLM scenes:", DRIVELM_SCENES or "not attached")

## 7. Check memory headroom before raising the tile count

CHAI render at 12 tiles per frame (9 on these images). Eager attention makes that
roughly four times the attention memory of 6 tiles, so confirm there is room
rather than discovering it two hours in.

If this reports comfortable headroom, use `--max-tiles 12` below — that is the
faithful setting and the one to report.

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"gpu {i}: {free/1e9:5.1f} GB free of {total/1e9:5.1f} GB")

heads, seq6, seq12 = 32, 3300, 6400          # InternLM2.5-7B attention heads
for name, seq in (("6 tiles", seq6), ("12 tiles", seq12)):
    gb = seq * seq * heads * 2 / 1e9
    print(f"{name:9s} seq~{seq:5d} -> eager attention matrix {gb:5.2f} GB per layer")

## 9f. A second model family, at ten times the sample size

Two limitations bound everything above: **one model family** (InternVL2.5) and
**23 scenes**. This stage addresses both at once.

* **Scenes** come from the full DriveLM release rather than CHAI's subset,
  built by `midas-build-scenes` and attached as part of the source dataset.
  One key frame per nuScenes scene, so the set stays visually diverse.
* **Model** is Qwen2.5-VL-3B-Instruct, a different architecture, tokenizer and
  image pipeline. If the attack holds here it is not an InternVL quirk.

**Labels first.** DriveLM scenes carry no benign action, and inventing one would
repeat the ground-truth problem of section 9b. They are derived for this model
by running it unattacked and taking the mode, which is CHAI's own procedure.
`midas-eval` refuses to run on unlabelled scenes rather than defaulting.

**Two settings that cost a session if wrong**, both carried over from a sibling
project that ran these models on the same hardware:

* the vision tower runs in **float32**; in float16 it emits NaN on frames
  carrying rendered text, which is every attacked frame here. `midas-eval`
  selects this automatically under a float16 language model;
* **transformers 4.49.x** is the only version where Qwen2.5-VL (needs >= 4.49)
  and InternVL2.5 (breaks on >= 4.50) both work.

In [ ]:
# Resolved in the setup cell, since the steering stage needs the same two.
if DRIVELM_SCENES is None:
    raise SystemExit(
        "no DriveLM scenes in the attached dataset. Build them locally and "
        "re-upload:  midas-build-scenes --limit 100 "
        "&& python scripts/kaggle_deploy.py upload --drivelm"
    )
print("model :", DRIVELM_MODEL)
print("scenes:", DRIVELM_SCENES)
print("count :", len(json.load(open(DRIVELM_SCENES))))

In [ ]:
# Scenes carry a benign action already, taken from DriveLM's record of what the
# human driver did. No label-derivation pass is needed, which saves about an
# hour; section 9f-b compares that label against the model's own vote.
DRIVELM_LIMIT = None
LIMIT_ARGS = ["--limit", DRIVELM_LIMIT] if DRIVELM_LIMIT else []
print("scene limit:", DRIVELM_LIMIT or "none (full set)")

# Which wordings to evaluate, and in what order. Generation-time knob so the
# 7B run can ask for only the two arms that matter without editing this file.
DRIVELM_STYLES = ['plain', 'descriptive']
print("styles:", DRIVELM_STYLES)

# A 7B in float16 is ~16.6 GB and does not fit one T4, so it is sharded over
# both. The 3B fits on a single card and sharding it only adds transfers.
SHARD_ARGS = ["--device-map", "auto"] if any(
    tag in DRIVELM_MODEL for tag in ("7B", "8B", "13B")
) else []
print("sharding:", SHARD_ARGS or "single GPU")

In [ ]:
# Baseline first, then each wording, saving after every arm so a session that
# runs out of time still yields the arms that finished.
for style in [None] + DRIVELM_STYLES:
    name = style or "baseline"
    print(f"\n{'=' * 70}\nDriveLM / {DRIVELM_MODEL.split('/')[-1]} / {name}\n{'=' * 70}",
          flush=True)
    cmd = ["midas-eval", "--split", DRIVELM_SCENES, "--model", DRIVELM_MODEL,
           "--backend", "qwen", "--save-long", *LIMIT_ARGS, *SHARD_ARGS,
           "--out", f"/kaggle/working/results/drivelm/{name}"]
    cmd += ["--baseline"] if style is None else ["--attack", "--style", style]
    started = time.time()
    subprocess.run(cmd, check=True)
    print(f"[{name}] {time.time() - started:.0f}s", flush=True)

In [ ]:
base = None
for f in glob.glob("/kaggle/working/results/drivelm/baseline/*.json"):
    base = json.load(open(f))["summary"]

print(f"{'condition':22s} {'ASR':>8s} {'CHAI rule':>10s} {'vs clean':>9s} "
      f"{'unreadable':>11s} {'n':>4s}")
if base:
    print(f"{'no sign':22s} {base['asr']:8.1%} {base['asr_chai']:10.1%} "
          f"{'--':>9s} {base['unreadable_rate']:11.1%} {base['n']:4d}")
for style in STYLES:
    files = glob.glob(f"/kaggle/working/results/drivelm/{style}/*.json")
    if not files:
        continue
    s = json.load(open(files[0]))["summary"]
    delta = f"{s['asr_chai'] - base['asr_chai']:+.1%}" if base else "-"
    print(f"{style:22s} {s['asr']:8.1%} {s['asr_chai']:10.1%} {delta:>9s} "
          f"{s['unreadable_rate']:11.1%} {s['n']:4d}")

if base:
    n = base["n"]
    print("")
    print(f"n = {n}, so one scene is {1 / n:.1%} and the 95% interval is about "
          f"+/-{98 / (n ** 0.5):.0f} points.")
    print("Compare with the 23-scene numbers above, where one scene was 6.7.")

### Activations for the steering defense

Collected in the same session, because loading a model is the expensive part and
the steering work (section 13) needs two readouts of it:

* **prompt-only** -- the open turn, before any token is generated. This is what
  the detection probe can actually see at inference, and fitting a gate anywhere
  else was worth 13 points of accuracy when we got it wrong before.
* **answer** -- teacher-forced at the answer token, both the benign and the
  attacker's action on the same image. This is the pair the steering direction
  is fitted on.

In [ ]:
for readout_args, label in (
    (["--prompt-only"], "prompt-only (detection gate)"),
    ([], "answer token (steering direction)"),
):
    print(f"\n{'=' * 70}\nactivations: {label}\n{'=' * 70}", flush=True)
    subprocess.run(
        ["midas-activations", "--split", DRIVELM_SCENES, "--model", DRIVELM_MODEL,
         "--backend", "qwen", "--phrases", "1", "--controls", "2",
         *LIMIT_ARGS, *readout_args,
         "--out", "/kaggle/working/results/drivelm/activations"],
        check=True,
    )
print(sorted(glob.glob("/kaggle/working/results/drivelm/activations/*.npz")))

## 11. Package the results

`/kaggle/working` persists between sessions, but download the archive anyway so
the results can be committed to the repository alongside the 2B numbers.

In [ ]:
import shutil, json, glob

shutil.make_archive("/kaggle/working/midas_8b_results", "zip",
                    "/kaggle/working/results")
print("archive written")
!du -sh /kaggle/working/midas_8b_results.zip

for f in sorted(glob.glob("/kaggle/working/results/runs_8b/*.json")):
    d = json.load(open(f))
    s = d["summary"]
    print(f"{d['label']:9s} {d['split']:6s} n={s['n']:<3} "
          f"ASR={s['asr']:6.1%}  benign={s['benign_accuracy']:6.1%}")